# File Connection Checks

Checks that file-level identifiers referenced across GTFS tables actually exist where expected.

In [10]:
from pathlib import Path
import sys

_current = Path.cwd().resolve()
for _candidate in [_current, *_current.parents]:
    if (_candidate / "data_validation" / "checks" / "commons.py").exists():
        _project_root = _candidate
        break
else:
    raise FileNotFoundError("data_validation/checks/commons.py not found.")

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from data_validation.gtfs_utils import (
    PATHWAYS_FILE,
    ROUTES_FILE,
    STOPS_FILE,
    STOP_TIMES_FILE,
    TRIPS_FILE,
    check_missing_files,
    print_file_disclaimer,
    load_from_stop_ids,
    load_route_ids,
    load_stop_ids,
    load_to_stop_ids,
    load_trip_ids,
)

In [15]:
def main() -> None:
    """Validate that stop_id, route_id, and trip_id references are consistent across files."""
    pathways_from_stop_ids = None
    pathways_to_stop_ids = None
    stop_times_stop_ids = None
    stops_stop_ids = None
    missing_pathways_from = None
    missing_pathways_to = None
    missing_stop_times = None
    trips_route_ids = None
    routes_route_ids = None
    missing_route_ids = None
    stop_times_trip_ids = None
    trips_trip_ids = None
    missing_trip_ids = None
    check_missing_files([PATHWAYS_FILE, STOP_TIMES_FILE, STOPS_FILE, TRIPS_FILE, ROUTES_FILE])

    pathways_from_stop_ids = load_from_stop_ids(PATHWAYS_FILE)
    pathways_to_stop_ids = load_to_stop_ids(PATHWAYS_FILE)
    stop_times_stop_ids = load_stop_ids(STOP_TIMES_FILE)
    stops_stop_ids = load_stop_ids(STOPS_FILE)

    print_file_disclaimer([
        (PATHWAYS_FILE, 'pathways'),
        (STOP_TIMES_FILE, 'stop_times'),
        (STOPS_FILE, 'stops'),
        (TRIPS_FILE, 'trips'),
        (ROUTES_FILE, 'routes'),
    ])

    missing_pathways_from = sorted(
        ids for ids in pathways_from_stop_ids if ids not in stops_stop_ids
    )
    missing_pathways_to = sorted(
        ids for ids in pathways_to_stop_ids if ids not in stops_stop_ids
    )
    missing_stop_times = sorted(
        ids for ids in stop_times_stop_ids if ids not in stops_stop_ids
    )

    print(f"\n----- All stop_id from 'pathways' and 'stop_times' exist in 'stops'? -----")
    if not missing_pathways_from:
        print(f" - All correct: no from_stop_id from 'pathways' is missing in 'stops'")
    else:
        print(f" - MISSING {len(missing_pathways_from)} from_stop_id from 'pathways' in 'stops':")
        for sid in missing_pathways_from:
            print("   -", sid)

    if not missing_pathways_to:
        print(f" - All correct: no to_stop_id from 'pathways' is missing in 'stops'")
    else:
        print(f" - MISSING {len(missing_pathways_to)} to_stop_id from 'pathways' in 'stops':")
        for sid in missing_pathways_to:
            print("   -", sid)

    if not missing_stop_times:
        print(f" - All correct: no stop_id from 'stop_times' is missing in 'stops'")
    else:
        print(f" - MISSING {len(missing_stop_times)} stop_id from 'stop_times' in 'stops':")
        for sid in missing_stop_times:
            print("   -", sid)

    print(f"\n----- All route_id from 'trips' exist in 'routes'? -----")
    trips_route_ids = load_route_ids(TRIPS_FILE)
    routes_route_ids = load_route_ids(ROUTES_FILE)
    missing_route_ids = sorted(ids for ids in trips_route_ids if ids not in routes_route_ids)

    print(f"Unique route_id in 'trips': {len(trips_route_ids)}")
    print(f"Unique route_id in 'routes': {len(routes_route_ids)}")

    if not missing_route_ids:
        print(f"All correct: all route_id present in 'trips' also appear in 'routes'.")
    else:
        print(
            f"MISSING {len(missing_route_ids)} route_id"
            f" (present in 'trips' but not in 'routes'):"
        )
        for rid in missing_route_ids:
            print(f"- {rid}")

    stop_times_trip_ids = load_trip_ids(STOP_TIMES_FILE)
    trips_trip_ids = load_trip_ids(TRIPS_FILE)
    missing_trip_ids = sorted(ids for ids in stop_times_trip_ids if ids not in trips_trip_ids)

    print(f"Unique trip_id in 'stop_times': {len(stop_times_trip_ids)}")
    print(f"Unique trip_id in 'trips': {len(trips_trip_ids)}")

    print(f"\n----- All trip_id from 'stop_times' exist in 'trips'? -----")
    if not missing_trip_ids:
        print(f"All correct: all trip_id present in 'stop_times' also appear in 'trips'.")
    else:
        print(
            f"MISSING {len(missing_trip_ids)} trip_id"
            f" (present in 'stop_times' but not in 'trips'):"
        )
        for tid in missing_trip_ids:
            print(f"- {tid}")

main()

Disclaimer: for coherence we will consider the next files from /Users/saradalmauguamis/Desktop/Mates/Cursos/Curs 2025-2026 (3r + 4t)/TFG/TFG/.src/gtfs/data:
 - pathways.txt as 'pathways' from 0_original
 - stop_times_cleaned.txt as 'stop_times' from 2_duplicated_trips
 - stops_subway.txt as 'stops' from 1_subway
 - trips_cleaned.txt as 'trips' from 2_duplicated_trips
 - routes_subway.txt as 'routes' from 1_subway

----- All stop_id from 'pathways' and 'stop_times' exist in 'stops'? -----
 - All correct: no from_stop_id from 'pathways' is missing in 'stops'
 - All correct: no to_stop_id from 'pathways' is missing in 'stops'
 - All correct: no stop_id from 'stop_times' is missing in 'stops'

----- All route_id from 'trips' exist in 'routes'? -----
Unique route_id in 'trips': 11
Unique route_id in 'routes': 11
All correct: all route_id present in 'trips' also appear in 'routes'.
Unique trip_id in 'stop_times': 10977
Unique trip_id in 'trips': 10977

----- All trip_id from 'stop_times' exi